In [1]:
import os
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import files

# --- 1. KAGGLE API SETUP ---
# Upload your kaggle.json file when prompted
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Please upload your kaggle.json file.")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

# --- 2. DOWNLOAD & UNZIP DATASET ---
# API command from the Kaggle dataset page
!kaggle datasets download -d dmaso01dsta/cisi-a-dataset-for-information-retrieval
!unzip -o cisi-a-dataset-for-information-retrieval.zip

# --- 3. LOADING DATA ---
# The CISI dataset uses a specific format (.W for text). We'll parse it into a list.
with open('CISI.ALL', 'r') as f:
    lines = f.readlines()

documents = []
current_text = ""
for line in lines:
    if line.startswith(".I"): # New document ID
        if current_text:
            documents.append(current_text.strip())
            current_text = ""
    elif line.startswith(".W"): # Start of the text body
        continue
    else:
        # Check if line is a metadata tag we should skip (like .T, .A, .X)
        if not line.startswith("."):
            current_text += line.replace('\n', ' ')
if current_text:
    documents.append(current_text.strip())

print(f"Loaded {len(documents)} documents.")

Please upload your kaggle.json file.


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/dmaso01dsta/cisi-a-dataset-for-information-retrieval
License(s): other
100% 759k/759k [00:00<00:00, 111MB/s]

Archive:  cisi-a-dataset-for-information-retrieval.zip
  inflating: CISI.ALL                
  inflating: CISI.QRY                
  inflating: CISI.REL                
Loaded 1460 documents.


In [15]:
# --- DATASET STRUCTURE ANALYSIS ---
# Convert the list of documents into a pandas DataFrame for better visualization
df = pd.DataFrame(documents, columns=['Document_Text'])

# 1. Basic Stats
print("--- Dataset Overview ---")
print(f"Total number of documents: {len(df)}") # Should be 1460 for CISI
print(f"Average length of documents (characters): {df['Document_Text'].str.len().mean():.2f}")

# 2. Display the 'Structure' (Head of the data)
print("\n--- First 5 Documents in the Collection ---")
# This displays the first 5 rows to show the format
print(df.head())

# 3. Text Sample (Detailed view of one document)
print("\n--- Sample Document Content (Index 0) ---")
print(documents[0])

--- Dataset Overview ---
Total number of documents: 1460
Average length of documents (characters): 1431.00

--- First 5 Documents in the Collection ---
                                       Document_Text
0  18 Editions of the Dewey Decimal Classificatio...
1  Use Made of Technical Libraries Slater, M. Thi...
2  Two Kinds of Power An Essay on Bibliographic C...
3  Systems Analysis of a University Library;  fin...
4  A Library Management Game: a report on a resea...

--- Sample Document Content (Index 0) ---
18 Editions of the Dewey Decimal Classifications Comaromi, J.P.    The present study is a history of the DEWEY Decimal Classification.  The first edition of the DDC was published in 1876, the eighteenth edition in 1971, and future editions will continue to appear as needed.  In spite of the DDC's long and healthy life, however, its full story has never been told.  There have been biographies of Dewey that briefly describe his system, but this is the first attempt to provide a detail

In [16]:
# --- 4. PREPROCESSING & CLEANING ---
nltk.download('stopwords')
nltk.download('wordnet')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Convert to lowercase and split
    words = text.lower().split()
    # Remove stopwords and lemmatize (e.g., 'running' -> 'run')
    cleaned = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return " ".join(cleaned)

# Apply cleaning to all documents
cleaned_docs = [clean_text(doc) for doc in documents]


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [17]:
import numpy as np
# --- 5. INDEXING & SEARCH FUNCTION ---
def search_with_scores(query):
    # 1. Preprocess the query
    query_cleaned = clean_text(query)

    # 2. Vectorization (TF-IDF)
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(cleaned_docs)
    query_vec = vectorizer.transform([query_cleaned])

    # 3. Calculate Cosine Similarity
    # This measures the mathematical 'angle' between the query and documents
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

    # 4. Show All Top Scores (Top 10)
    # We use argsort to find the indices of the highest scores
    top_indices = scores.argsort()[-10:][::-1]

    print(f"--- DETAILED SIMILARITY RANKING FOR: '{query}' ---")
    print(f"{'Rank':<5} | {'Doc ID':<8} | {'Score (%)':<10}")
    print("-" * 30)

    for i, idx in enumerate(top_indices, 1):
        score_pct = scores[idx] * 100 # Converting to percentage for clarity
        if score_pct > 0:
            print(f"{i:<5} | {idx:<8} | {score_pct:>8.2f}%")

    # 5. Highlight the Most Relevant Document
    best_idx = top_indices[0]
    if scores[best_idx] > 0:
        print("\n" + "="*50)
        print("⭐ MOST RELEVANT DOCUMENT FOUND ⭐")
        print(f"Document ID: {best_idx}")
        print(f"Confidence Score: {scores[best_idx]:.4f}")
        print("-" * 50)
        print(f"CONTENT:\n{documents[best_idx]}")
        print("="*50)
    else:
        print("\nNo relevant documents found for this query.")

# Run the search
user_query = input("Enter your search topic: ")
search_with_scores(user_query)

Enter your search topic: computer science
--- DETAILED SIMILARITY RANKING FOR: 'computer science' ---
Rank  | Doc ID   | Score (%) 
------------------------------
1     | 1029     |    34.86%
2     | 324      |    31.58%
3     | 1141     |    31.21%
4     | 455      |    27.84%
5     | 691      |    26.50%
6     | 468      |    25.89%
7     | 598      |    24.91%
8     | 325      |    24.43%
9     | 367      |    22.60%
10    | 1369     |    22.59%

⭐ MOST RELEVANT DOCUMENT FOUND ⭐
Document ID: 1029
Confidence Score: 0.3486
--------------------------------------------------
CONTENT:
Little Science, Big Science De Solla Price, D.J.   Pegram lecturers are supposed to talk about science and its place in society.  The ordinary way of doing this would be either to talk popular science or to adopt one of the various styles in humanistic discussion of the reactions between men and science.  Previous lecturers in this series have given accounts of the content of space science and made excursio